# Exploratory Visualization: Olympics and Housing (US & UK Markets)

Interactive Altair visualizations exploring Olympic bid/games cycles and housing markets.

## Data Sources

- **US:** FRED S&P CoreLogic Case-Shiller (Atlanta, LA, Chicago, NY)
- **UK:** ONS House Price Index (London)
- **Context:** COHRE 2007 | Comitê Popular 2014 | IOC records

## Olympic Events
- Atlanta 1996 (bid 1990) | London 2012 (bid 2005) | LA 2028 (bid 2017)

## Focus

This notebook focuses on one hypothesis and evaluates it with three visuals.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()


def load_fred_series(series_id: str) -> pd.DataFrame:
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    df = pd.read_csv(url)
    df.columns = ['date', 'index_value']
    df['date'] = pd.to_datetime(df['date'])
    return df.dropna(subset=['index_value']).copy()


def load_london_olympic_boroughs() -> pd.DataFrame:
    df = pd.read_csv('data/london_olympic_boroughs_ukhpi.csv')
    df['date'] = pd.to_datetime(df['date'])
    df['index_value'] = pd.to_numeric(df['avg_price_all_property_types'], errors='coerce')
    df = df.dropna(subset=['index_value']).copy()
    # Aggregate the five host boroughs into one London monthly series.
    return df.groupby('date', as_index=False)['index_value'].mean()


In [2]:
# Load and prep data

series_map = {'Atlanta': 'ATXRSA', 'Los Angeles': 'LXXRSA', 'Chicago': 'CHXRSA', 'New York': 'NYXRSA'}

frames = []

for city, sid in series_map.items():

    tmp = load_fred_series(sid)

    tmp['city'], tmp['region'] = city, 'US'

    frames.append(tmp)



uk = load_london_olympic_boroughs()

uk['city'], uk['region'] = 'London', 'UK'

frames.append(uk)



housing = pd.concat(frames, ignore_index=True)

housing['year'] = housing['date'].dt.year

annual = housing.groupby(['city', 'year'], as_index=False)['index_value'].mean().sort_values(['city', 'year'])

annual['yoy_pct'] = annual.groupby('city')['index_value'].pct_change() * 100



events = pd.DataFrame({'city': ['Atlanta', 'London', 'Los Angeles'], 'bid_year': [1990, 2005, 2017], 'games_year': [1996, 2012, 2028]})

annual_evt = annual.merge(events, on='city', how='left')



displacement = pd.DataFrame({'city': ['Seoul', 'Atlanta', 'Beijing', 'London', 'Rio'], 'games_year': [1988, 1996, 2008, 2012, 2016], 'estimated_displaced': [720000, 15000, 1500000, 2000, 22000], 'source': ['COHRE 2007', 'COHRE 2007', 'COHRE 2007', 'COHRE 2007', 'Comite Popular 2014']})



print('✓ Data loaded')

print('London source: real downloaded borough CSV')

print('Cities:', sorted(housing['city'].unique()))

print('London monthly rows:', len(uk))


✓ Data loaded
London source: real downloaded borough CSV
Cities: ['Atlanta', 'Chicago', 'London', 'Los Angeles', 'New York']
London monthly rows: 316


## Hypothesis 1: Bid-to-Games Price Acceleration

**Hypothesis:** Housing index levels and growth rates accelerate during the Olympic bid-to-games window, especially in host markets.

In [15]:
# H1.1: Housing Index with Olympic Markers (Altair line chart)
chart = alt.Chart(annual_evt).mark_line(point=True, size=2).encode(
    x=alt.X('year:Q', title='Year'),
    y=alt.Y('index_value:Q', title='Index Value'),
    color=alt.Color('city:N', legend=alt.Legend(title='City')),
    tooltip=['year:Q', 'city:N', alt.Tooltip('index_value:Q', format='.1f')]
)

bid_rule = alt.Chart(pd.DataFrame({'year': [1990, 2005, 2017]})).mark_rule(stroke='red', strokeDash=[3,3], size=2).encode(x='year:Q')
games_rule = alt.Chart(pd.DataFrame({'year': [1996, 2012]})).mark_rule(stroke='green', strokeDash=[3,3], size=2).encode(x='year:Q')

h1_1 = (chart + bid_rule + games_rule).properties(width=900, height=400, title='H1.1: Real Housing Index by City')
h1_1.display()

alt.LayerChart(...)

### H1 - Visual 1: Index Level with Olympic Markers

**What's informative about this view:**
- Shows long-run index movement for each city in a single frame.
- Makes bid and games years explicit with vertical markers.
- Helps compare whether London behaves differently from US cities.

**What could be improved about this view:**
- Index levels are not directly comparable across different baselines.
- Add normalization or faceting to reduce scale effects.
- Add confidence/context bands (macro cycles) for interpretation.

In [16]:
# H1.2: YoY Growth (Altair line chart)

h1_2 = alt.Chart(annual_evt).mark_line(point=True, size=2).encode(

    x=alt.X('year:Q', title='Year'),

    y=alt.Y('yoy_pct:Q', title='YoY Change (%)'),

    color=alt.Color('city:N'),

    tooltip=['year:Q', 'city:N', alt.Tooltip('yoy_pct:Q', format='.2f')]

).properties(width=900, height=400, title='H1.2: Year-over-Year Growth')

h1_2.display()


alt.Chart(...)

### H1 - Visual 2: Year-over-Year Growth

**What's informative about this view:**
- Focuses on acceleration/deceleration, not just raw levels.
- Better for spotting potential event-window shifts.
- Highlights mixed signal across cities rather than one uniform pattern.

**What could be improved about this view:**
- Volatility can obscure structural changes.
- Add smoothed trend lines (for example 3-year rolling mean).
- Add pre/post averages and statistical test annotations.

In [17]:
# H1.3: Indexed Growth 2005=100 (Altair line chart)
idx = annual_evt.copy()
base = idx[idx['year']==2005][['city', 'index_value']].rename(columns={'index_value': 'base_2005'})
idx = idx.merge(base, on='city', how='left').dropna(subset=['base_2005'])
idx['indexed_2005'] = (idx['index_value'] / idx['base_2005']) * 100

chart = alt.Chart(idx).mark_line(point=True, size=2).encode(
    x=alt.X('year:Q', title='Year'),
    y=alt.Y('indexed_2005:Q', title='Index (2005 = 100)'),
    color=alt.Color('city:N'),
    tooltip=['year:Q', 'city:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

ref = alt.Chart(pd.DataFrame({'indexed_2005': [100]})).mark_rule(stroke='black', strokeDash=[2,2]).encode(y='indexed_2005:Q')
h1_3 = (chart + ref).properties(width=900, height=400, title='H1.3: Indexed Growth (2005 = 100)')
h1_3.display()

alt.LayerChart(...)

### H1 - Visual 3: Indexed Growth (2005 = 100)

**What's informative about this view:**
- Normalization improves cross-city comparability.
- Frames relative growth from a common anchor year.
- Useful for discussing London around bid year context.

**What could be improved about this view:**
- 2005 anchor may advantage/disadvantage some markets.
- Add sensitivity checks with alternate base years.
- Include host-borough-level London line as a refinement.

## London Zoom-in: Host Boroughs vs London Baseline

The city-level view mixes London in with US markets on different scales. To test the bid-to-games hypothesis more directly, zoom into London and compare the **five host boroughs** (Newham, Tower Hamlets, Hackney, Waltham Forest, Greenwich) against the **London-wide baseline**.

If the Olympic bid/games window disproportionately affected host neighbourhoods, we would expect the host-borough series to diverge upward from the London baseline after the 2005 bid award.


In [3]:
# H1.4: London host boroughs vs London-wide baseline, indexed to 2005 = 100

# Host-borough aggregate: already prepped upstream as `uk` (mean avg price across 5 host boroughs, monthly)
host = uk[['date', 'index_value']].copy()
host['series'] = 'Host boroughs (avg of 5)'

# London-wide baseline from Land Registry UKHPI regional file.
# Columns of interest: "Average price All property types" and "Pivotable date".
london_base = pd.read_csv(
    'data/london_region_ukhpi.csv',
    usecols=['Average price All property types', 'Pivotable date']
).rename(columns={'Average price All property types': 'index_value', 'Pivotable date': 'date'})
london_base['date'] = pd.to_datetime(london_base['date'])
london_base = london_base.dropna(subset=['index_value']).sort_values('date')
london_base['series'] = 'London (whole region)'

london_cmp = pd.concat([host[['date', 'index_value', 'series']], london_base[['date', 'index_value', 'series']]], ignore_index=True)
london_cmp['year'] = london_cmp['date'].dt.year

# Index each series to its 2005 mean = 100
base_2005 = (
    london_cmp[london_cmp['year'] == 2005]
    .groupby('series', as_index=False)['index_value'].mean()
    .rename(columns={'index_value': 'base_2005'})
)
london_cmp = london_cmp.merge(base_2005, on='series', how='left')
london_cmp['indexed_2005'] = (london_cmp['index_value'] / london_cmp['base_2005']) * 100

# Keep a sensible window (2000 onward, up to latest available)
london_cmp = london_cmp[london_cmp['year'] >= 2000]

lines = alt.Chart(london_cmp).mark_line(size=2).encode(
    x=alt.X('date:T', title='Year'),
    y=alt.Y('indexed_2005:Q', title='Price index (2005 = 100)'),
    color=alt.Color('series:N', legend=alt.Legend(title='Series'),
                    scale=alt.Scale(range=['#d62728', '#1f77b4'])),
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

bid_rule = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2005-07-06')], 'label': ['Bid awarded (Jul 2005)']})).mark_rule(
    stroke='red', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

games_rule = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2012-07-27')], 'label': ['Games open (Jul 2012)']})).mark_rule(
    stroke='green', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

ref_100 = alt.Chart(pd.DataFrame({'indexed_2005': [100]})).mark_rule(stroke='black', strokeDash=[2, 2]).encode(y='indexed_2005:Q')

h1_4 = (lines + bid_rule + games_rule + ref_100).properties(
    width=900, height=400,
    title='H1.4: London host boroughs vs London-wide baseline (indexed to 2005 = 100)'
)
h1_4.display()


alt.LayerChart(...)

### H1 - Visual 4: London Host Boroughs vs London Baseline

**What's informative about this view:**
- Isolates London so the comparison isn't distorted by US market scales.
- Anchors both series at 2005 = 100, the year London won the bid, so any divergence after the red rule is directly interpretable.
- The gap between the red and blue lines is the "host-borough premium" over the London-wide market.

**What could be improved about this view:**
- Averaging five boroughs hides within-group variation — Newham and Hackney may behave very differently.
- Doesn't isolate the counterfactual (what host boroughs would have done without the Games).
- A follow-up could split by individual borough or add a non-host inner-London borough as a control.


### Refinement: Each Host Borough vs London Baseline

The averaged host-borough line barely diverges from the London-wide series. But averaging can wash out effects that hit some neighbourhoods harder than others. The next chart keeps the five host boroughs **separated**, with London-wide as a thick reference line, so we can see whether any individual borough breaks away after the 2005 bid.


In [4]:
# H1.5: Each host borough vs London-wide baseline, indexed to 2005 = 100

# Per-borough series from the already-downloaded olympic boroughs CSV
boroughs = pd.read_csv('data/london_olympic_boroughs_ukhpi.csv')
boroughs['date'] = pd.to_datetime(boroughs['date'])
boroughs = boroughs.rename(columns={'avg_price_all_property_types': 'index_value'})
boroughs['index_value'] = pd.to_numeric(boroughs['index_value'], errors='coerce')
boroughs = boroughs.dropna(subset=['index_value'])[['date', 'borough', 'index_value']]
boroughs = boroughs.rename(columns={'borough': 'series'})

# London-wide baseline (already loaded as london_base above; rebuild defensively)
london_ref = london_base[['date', 'index_value']].copy()
london_ref['series'] = 'London (whole region)'

per_borough = pd.concat([boroughs, london_ref], ignore_index=True)
per_borough['year'] = per_borough['date'].dt.year

# Index each series to its own 2005 mean = 100
base_2005_pb = (
    per_borough[per_borough['year'] == 2005]
    .groupby('series', as_index=False)['index_value'].mean()
    .rename(columns={'index_value': 'base_2005'})
)
per_borough = per_borough.merge(base_2005_pb, on='series', how='left')
per_borough['indexed_2005'] = (per_borough['index_value'] / per_borough['base_2005']) * 100
per_borough = per_borough[per_borough['year'] >= 2000]

# Split so we can style the London baseline differently (thick black)
pb_boroughs = per_borough[per_borough['series'] != 'London (whole region)']
pb_london = per_borough[per_borough['series'] == 'London (whole region)']

borough_names = sorted(pb_boroughs['series'].unique().tolist())

borough_lines = alt.Chart(pb_boroughs).mark_line(size=2).encode(
    x=alt.X('date:T', title='Year'),
    y=alt.Y('indexed_2005:Q', title='Price index (2005 = 100)'),
    color=alt.Color('series:N',
                    scale=alt.Scale(domain=borough_names, scheme='category10'),
                    legend=alt.Legend(title='Host borough')),
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

london_line = alt.Chart(pb_london).mark_line(size=4, color='black', strokeDash=[6, 3]).encode(
    x='date:T',
    y='indexed_2005:Q',
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

bid_rule_pb = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2005-07-06')], 'label': ['Bid awarded (Jul 2005)']})).mark_rule(
    stroke='red', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

games_rule_pb = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2012-07-27')], 'label': ['Games open (Jul 2012)']})).mark_rule(
    stroke='green', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

ref_100_pb = alt.Chart(pd.DataFrame({'indexed_2005': [100]})).mark_rule(stroke='gray', strokeDash=[2, 2]).encode(y='indexed_2005:Q')

h1_5 = (borough_lines + london_line + bid_rule_pb + games_rule_pb + ref_100_pb).properties(
    width=900, height=450,
    title='H1.5: Each host borough vs London-wide baseline (dashed black, indexed to 2005 = 100)'
)
h1_5.display()


alt.LayerChart(...)

### H1 - Visual 5: Per-Borough View

**What's informative about this view:**
- Individual borough lines reveal the spread that the average in H1.4 hid.
- The dashed black London-wide line is a shared reference — anything above it after 2005 is a borough outpacing the wider market.
- Colour + tooltips let the reader hover to identify which borough diverged and when.

**What could be improved about this view:**
- Five coloured lines plus the baseline can get busy — small multiples (one panel per borough) would separate signals more cleanly.
- Still no true counterfactual: divergence could reflect gentrification pressure independent of the Games.
- Adding a non-host inner-London borough (e.g. Islington or Southwark) as a comparison would strengthen the causal read.


### Refinement: Excess Growth Over London

H1.4 and H1.5 both show borough lines *hugging* the London baseline — which is honest but visually inconclusive. The signal we care about is the **gap**: how much did each host borough outpace (or lag) the London-wide market after the bid?

The next chart plots `borough_index − london_index` at every month (both indexed to 2005 = 100). A borough sitting at +5 means it has grown five index points *more* than London since the bid. This is the same "excess return" framing Kavetsos (2012) used when he found a 2.1–3.3% host-borough premium in the 24 months after the bid award — we can eyeball that number directly here.


In [5]:
# H1.6: Excess growth over London — each host borough minus London-wide baseline

# Start from `per_borough` (already indexed to 2005 = 100 for every series).
# Pivot so each row is a date with borough columns + a London column, then subtract.
pivoted = (
    per_borough[['date', 'series', 'indexed_2005']]
    .pivot_table(index='date', columns='series', values='indexed_2005')
    .sort_index()
)
london_col = 'London (whole region)'
excess = pivoted.drop(columns=[london_col]).sub(pivoted[london_col], axis=0).reset_index()
excess_long = excess.melt(id_vars='date', var_name='borough', value_name='excess_over_london')
excess_long = excess_long.dropna(subset=['excess_over_london'])
excess_long = excess_long[excess_long['date'].dt.year >= 2000]

# Kavetsos-style stat: mean excess in the 24 months AFTER the July 2005 bid award,
# vs. mean excess in the 24 months BEFORE. Print it as a sanity check.
bid_date = pd.Timestamp('2005-07-06')
pre = excess_long[(excess_long['date'] >= bid_date - pd.DateOffset(months=24)) & (excess_long['date'] < bid_date)]
post = excess_long[(excess_long['date'] >= bid_date) & (excess_long['date'] < bid_date + pd.DateOffset(months=24))]
kavetsos_check = (
    post.groupby('borough')['excess_over_london'].mean()
    - pre.groupby('borough')['excess_over_london'].mean()
).round(2).sort_values(ascending=False)
print('Change in excess-over-London (index points), 24 months post-bid minus 24 months pre-bid:')
print(kavetsos_check.to_string())

# Chart: one line per borough, zero-rule = "same growth as London".
excess_lines = alt.Chart(excess_long).mark_line(size=2).encode(
    x=alt.X('date:T', title='Year'),
    y=alt.Y('excess_over_london:Q', title='Excess index over London (2005 = 100 base)'),
    color=alt.Color('borough:N',
                    scale=alt.Scale(domain=borough_names, scheme='category10'),
                    legend=alt.Legend(title='Host borough')),
    tooltip=['date:T', 'borough:N', alt.Tooltip('excess_over_london:Q', format='+.1f')]
)

zero_rule = alt.Chart(pd.DataFrame({'excess_over_london': [0]})).mark_rule(
    stroke='black', strokeDash=[2, 2]
).encode(y='excess_over_london:Q')

bid_rule_ex = alt.Chart(pd.DataFrame({'date': [bid_date], 'label': ['Bid awarded (Jul 2005)']})).mark_rule(
    stroke='red', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

games_rule_ex = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2012-07-27')], 'label': ['Games open (Jul 2012)']})).mark_rule(
    stroke='green', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

h1_6 = (excess_lines + zero_rule + bid_rule_ex + games_rule_ex).properties(
    width=900, height=420,
    title='H1.6: Host-borough excess growth over London (2005 = 100 base). Above zero = outpacing London.'
)
h1_6.display()


Change in excess-over-London (index points), 24 months post-bid minus 24 months pre-bid:
borough
Hackney           4.77
Tower Hamlets     0.85
Greenwich         0.17
Waltham Forest    0.14
Newham           -1.24


alt.LayerChart(...)

### H1 - Visual 6: Excess Growth Over London

**What the numbers say** (24-month post-bid mean − 24-month pre-bid mean, in index points):

| Borough | Δ excess over London |
|---|---:|
| Hackney | **+4.77** |
| Tower Hamlets | +0.85 |
| Greenwich | +0.17 |
| Waltham Forest | +0.14 |
| Newham | -1.24 |

**What's informative about this view:**
- The zero rule makes the "same as London" counterfactual explicit — every line above it is a borough being repriced faster than the wider market.
- The single-borough concentration is the actual finding: the Games-related premium didn't spread evenly across all five host boroughs. Hackney is the story.
- Kavetsos (2012) reported a **2.1–3.3% host-borough premium** in the 24 months after the bid on transaction-level hedonics. Our aggregate UKHPI-based read is in the same order of magnitude but *concentrated* in one borough rather than averaged across five.

**What could be improved about this view:**
- Aggregate borough-average prices ignore mix effects. Two boroughs can have the same UKHPI trajectory while selling very different property types — Price Paid Data would let us slice by property type (flat vs terrace) and new-build flag to see which sub-market absorbed the premium.
- The 24-month window is a convention borrowed from Kavetsos. A longer post-window (out to the 2012 Games) shows Newham catching up sharply — Olympic Park construction is directly next door — so the pre-Games effect and the actual-Games effect are different stories.
- No counterfactual borough: Islington or Southwark (inner-London, non-host) would strengthen the causal read.



### Refinement: Adding Controls — Islington and England

The excess-growth chart flags Hackney but leaves an obvious gap: how do we know the "gap" is Olympic-driven and not just inner-London gentrification or national market movement?

Two natural controls to add:

- **Islington** — an inner-London borough that is *not* a host but sits next to the host cluster and shares similar demographic/pricing pressures. If Islington also outpaced London-wide by the same margin as Hackney, the "Olympic premium" is really an inner-London-gentrification premium.
- **England (whole country)** — a national baseline. If London-wide already outpaced England after 2005, the earlier charts were understating how much host boroughs pulled ahead of a "normal" market.

Data source for both: HM Land Registry UK HPI monthly series, fetched from the official linked-data API and saved to `data/england_ukhpi.csv` and `data/islington_ukhpi.csv`.


In [6]:
# H1.7: Host boroughs + London vs Islington (non-host control) + England (national), indexed to 2005 = 100

def load_ukhpi_reference(path: str, label: str) -> pd.DataFrame:
    """Load a saved UKHPI CSV (date, avg_price_all_property_types, ...) into the common schema."""
    df = pd.read_csv(path, parse_dates=['date'])
    df = df.rename(columns={'avg_price_all_property_types': 'index_value'})
    df['series'] = label
    return df[['date', 'series', 'index_value']]

england_ref = load_ukhpi_reference('data/england_ukhpi.csv', 'England (national baseline)')
islington_ref = load_ukhpi_reference('data/islington_ukhpi.csv', 'Islington (non-host inner London)')

# Reuse the already-built pieces
# - boroughs (5 host boroughs, per-series)
# - london_base (London whole region), given `series` column below

london_ref_v2 = london_base[['date', 'index_value']].copy()
london_ref_v2['series'] = 'London (whole region)'

controls = pd.concat([
    boroughs[['date', 'series', 'index_value']],
    london_ref_v2[['date', 'series', 'index_value']],
    islington_ref,
    england_ref,
], ignore_index=True)
controls['year'] = controls['date'].dt.year

# Index each series to its own 2005 mean = 100 (only keep series that have 2005 data)
base_2005_ctrl = (
    controls[controls['year'] == 2005]
    .groupby('series', as_index=False)['index_value'].mean()
    .rename(columns={'index_value': 'base_2005'})
)
controls = controls.merge(base_2005_ctrl, on='series', how='inner')
controls['indexed_2005'] = (controls['index_value'] / controls['base_2005']) * 100
controls = controls[controls['year'] >= 2000]

# Split into three visual roles
host_names = sorted(boroughs['series'].unique().tolist())
host_rows = controls[controls['series'].isin(host_names)]
london_rows = controls[controls['series'] == 'London (whole region)']
control_rows = controls[controls['series'].isin(['Islington (non-host inner London)', 'England (national baseline)'])]

host_layer = alt.Chart(host_rows).mark_line(size=2).encode(
    x=alt.X('date:T', title='Year'),
    y=alt.Y('indexed_2005:Q', title='Price index (2005 = 100)'),
    color=alt.Color('series:N',
                    scale=alt.Scale(domain=host_names, scheme='category10'),
                    legend=alt.Legend(title='Host borough')),
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

london_layer = alt.Chart(london_rows).mark_line(size=4, color='black', strokeDash=[6, 3]).encode(
    x='date:T', y='indexed_2005:Q',
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

control_layer = alt.Chart(control_rows).mark_line(size=3).encode(
    x='date:T', y='indexed_2005:Q',
    color=alt.Color('series:N',
                    scale=alt.Scale(
                        domain=['Islington (non-host inner London)', 'England (national baseline)'],
                        range=['#8c564b', '#7f7f7f']),
                    legend=alt.Legend(title='Control series')),
    strokeDash=alt.StrokeDash('series:N',
                              scale=alt.Scale(
                                  domain=['Islington (non-host inner London)', 'England (national baseline)'],
                                  range=[[1, 0], [2, 4]]),
                              legend=None),
    tooltip=['date:T', 'series:N', alt.Tooltip('indexed_2005:Q', format='.1f')]
)

bid_rule_v3 = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2005-07-06')], 'label': ['Bid awarded']})).mark_rule(
    stroke='red', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

games_rule_v3 = alt.Chart(pd.DataFrame({'date': [pd.Timestamp('2012-07-27')], 'label': ['Games open']})).mark_rule(
    stroke='green', strokeDash=[4, 3], size=2
).encode(x='date:T', tooltip='label:N')

ref_100_v3 = alt.Chart(pd.DataFrame({'indexed_2005': [100]})).mark_rule(stroke='gray', strokeDash=[2, 2]).encode(y='indexed_2005:Q')

h1_7 = (host_layer + london_layer + control_layer + bid_rule_v3 + games_rule_v3 + ref_100_v3).resolve_scale(
    color='independent', strokeDash='independent'
).properties(
    width=900, height=460,
    title='H1.7: Host boroughs + London + Islington (non-host control) + England (national), indexed to 2005 = 100'
)
h1_7.display()

# Quick numeric read: mean indexed value in the 24-month post-bid window
post_bid = controls[(controls['date'] >= pd.Timestamp('2005-07-01')) &
                    (controls['date'] < pd.Timestamp('2007-07-01'))]
print('Mean indexed value (2005=100), 24 months after bid award:')
print(post_bid.groupby('series')['indexed_2005'].mean().round(2).sort_values(ascending=False).to_string())


alt.LayerChart(...)

Mean indexed value (2005=100), 24 months after bid award:
series
Islington (non-host inner London)    111.38
Hackney                              110.99
Tower Hamlets                        109.04
London (whole region)                107.95
Waltham Forest                       107.68
England (national baseline)          107.36
Greenwich                            106.59
Newham                               105.99


### H1 - Visual 7: The Controls Kill the Story

**What the numbers say** (mean price index, 24 months after the July 2005 bid, all series set to 100 at 2005):

| Series | Mean index (2005-07 → 2007-06) |
|---|---:|
| **Islington (non-host inner London)** | **111.4** |
| Hackney (host) | 111.0 |
| Tower Hamlets (host) | 109.0 |
| London (whole region) | 108.0 |
| Waltham Forest (host) | 107.7 |
| England (national baseline) | 107.4 |
| Greenwich (host) | 106.6 |
| Newham (host) | 106.0 |

**What's informative about this view:**
- Islington — a non-host inner-London borough — grew *more* than every host borough in the two years after the bid award. That reframes the earlier Hackney result: the +4.8-point excess over London wasn't an Olympic premium, it was a generic inner-London gentrification premium that Islington shared without hosting anything.
- London-wide (108) outpaced England (107) by only about one index point in the same window, so the national market was moving at broadly the same speed.
- Three host boroughs (Waltham Forest, Greenwich, Newham) grew *slower* than the national English baseline. Nothing about hosting looks like a price accelerant here.

**What could be improved about this view:**
- One non-host borough (Islington) is a single-observation control. A synthetic control built from several similar non-host inner-London boroughs would be more defensible.
- The 24-month window ends in mid-2007, right before the global financial crisis. Extending to 2012 (Games year) or 2015 lets construction-phase effects show up, but also mixes in the post-2008 recovery — which affected boroughs unevenly.
- UKHPI is mix-adjusted only imperfectly at borough level. Price Paid Data joined to property attributes would give a cleaner hedonic comparison.

**Bottom line for H1:** The bid-to-games acceleration hypothesis does not survive a proper counterfactual on this data. What looked like an Olympic effect in H1.4–H1.6 is largely an inner-London-gentrification effect that a non-host neighbour shared. This is a useful negative finding to carry into the final report.


## Conclusion

### Single Hypothesis Tested
**Hypothesis:** Housing index levels and growth rates accelerate during the Olympic bid-to-games window, especially in host markets.

### Visual 1: Index Level with Olympic Markers
Shows long-run housing index trajectories and the timing of bid/games years.

### Visual 2: Year-over-Year Growth
Highlights acceleration/deceleration dynamics and whether shifts align with event windows.

### Visual 3: Indexed Growth (2005 = 100)
Normalizes city paths to improve cross-city comparability from a common base year.

### Overall Takeaway
The three visuals provide exploratory evidence of period-specific changes, but patterns remain mixed across cities. The strongest support for the hypothesis appears in selected host-market windows rather than uniformly across all markets.

### Data Sources
- US: FRED S&P CoreLogic Case-Shiller (Atlanta, Los Angeles, Chicago, New York)
- UK: ONS House Price Index (London host borough aggregate)
- Olympic context years from IOC records